# `pipeline/03_training/train_lora_sampler_ablation.ipynb`

Canonical training notebook for sampler ablation experiments:
- random batching
- grouped retrieval batching
- mixed grouped + random batching

This notebook is intended for new controlled experiments without modifying archival notebooks.

In [ ]:
# pip install -q -U transformers datasets peft accelerate sentencepiece evaluate bert-score rouge-score

In [ ]:
import os
import gc
import json
import math
import time
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import load_from_disk
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)

In [ ]:
# ---------- Manifest ----------
ROOT = "/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research"
MANIFEST_PATH = f"{ROOT}/manifests/bootstrap_manifest.json"

if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(
        f"bootstrap_manifest.json not found at: {MANIFEST_PATH}\n"
        "Run the bootstrap notebook first."
    )

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

DIRS = manifest["dirs"]

def _safe_makedirs(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def _now_utc() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

print("Loaded manifest created_at_utc:", manifest.get("created_at_utc"))
print("ROOT:", manifest.get("root", ROOT))
print("Experiments dir:", DIRS["experiments"])

In [ ]:
# ---------- Experiment identity ----------
EXP_ID = "exp_006_grouped_mixed_lora"
METHOD = "grouped_mixed_lora"

# ---------- Sampler strategy ----------
SAMPLER_MODE = "mixed"   # one of: "random", "grouped", "mixed"
GROUPED_FRACTION = 0.80  # used only when SAMPLER_MODE == "mixed"

# ---------- Dataset ----------
DATASET_NAME = "dolly_small_1k"   # scale to dolly_15k later
SPLIT_NAME = "train"
INSTRUCTION_FIELD = "instruction"
CONTEXT_FIELD = "context"
RESPONSE_FIELD = "response"

# ---------- Model ----------
BASE_MODEL_DIRNAME = "flan-t5-small"
BASE_MODEL_PATH = f"{DIRS['shared_models_base']}/{BASE_MODEL_DIRNAME}"

# ---------- Retrieval assets ----------
EMBED_MODEL_DIRNAME = "all-MiniLM-L6-v2"
BUNDLE_NAME = f"{DATASET_NAME}__{EMBED_MODEL_DIRNAME}"
BUNDLE_DIR = f"{DIRS['shared_indexes_faiss']}/{BUNDLE_NAME}"
TOP_K = 32

# ---------- Tokenization ----------
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256
CACHE_TOKENIZED = True
TOKEN_CACHE_VERSION = "tok_with_rawidx_v1"

# ---------- Train/Eval split ----------
EVAL_RATIO = 0.10
SEED = 42

# ---------- Training ----------
EPOCHS = 20
PER_DEVICE_BATCH = 8
GRAD_ACCUM = 1
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
LOGGING_STEPS = 20
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 3

# ---------- Precision ----------
FP16 = False
BF16 = False

# ---------- LoRA ----------
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q", "v"]

assert SAMPLER_MODE in {"random", "grouped", "mixed"}
if SAMPLER_MODE == "mixed":
    assert 0.0 < GROUPED_FRACTION < 1.0, "GROUPED_FRACTION must be in (0, 1) for mixed mode."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("EXP_ID:", EXP_ID)
print("METHOD:", METHOD)
print("SAMPLER_MODE:", SAMPLER_MODE)
print("DEVICE:", DEVICE)
print("BASE_MODEL_PATH:", BASE_MODEL_PATH)
print("BUNDLE_DIR:", BUNDLE_DIR)

In [ ]:
# ---------- Output directories ----------
RUN_ROOT = f"{DIRS['experiments']}/{EXP_ID}"
OUT_CHECKPOINTS = f"{RUN_ROOT}/checkpoints"
OUT_LOGS = f"{RUN_ROOT}/logs"
OUT_PLOTS = f"{RUN_ROOT}/plots"
OUT_TABLES = f"{RUN_ROOT}/tables"
OUT_ADAPTER = f"{RUN_ROOT}/adapter"

for p in [RUN_ROOT, OUT_CHECKPOINTS, OUT_LOGS, OUT_PLOTS, OUT_TABLES, OUT_ADAPTER]:
    _safe_makedirs(p)

RUN_MANIFEST_PATH = f"{RUN_ROOT}/run_manifest.json"
RUN_SUMMARY_PATH = f"{RUN_ROOT}/run_summary.json"
METRICS_JSONL_PATH = f"{OUT_LOGS}/metrics.jsonl"
LOSS_PLOT_PATH = f"{OUT_PLOTS}/loss_curve.png"
EXPERIMENT_CSV_PATH = f"{DIRS['experiments']}/experiment_logs/experiment_logs.csv"
_safe_makedirs(os.path.dirname(EXPERIMENT_CSV_PATH))

print("RUN_ROOT:", RUN_ROOT)
print("EXPERIMENT_CSV_PATH:", EXPERIMENT_CSV_PATH)

In [ ]:
# ---------- Dataset load + deterministic split ----------
dataset_path = f"{DIRS['shared_datasets_raw']}/{DATASET_NAME}"
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found: {dataset_path}")

ds = load_from_disk(dataset_path)
raw = ds[SPLIT_NAME]

required = {INSTRUCTION_FIELD, RESPONSE_FIELD}
missing = [c for c in required if c not in raw.column_names]
if missing:
    raise KeyError(f"Missing columns in dataset: {missing}. Available: {raw.column_names}")

raw = raw.map(lambda ex, idx: {"raw_idx": int(idx)}, with_indices=True)

base = raw.shuffle(seed=SEED)
n = len(base)
n_eval = max(1, int(n * EVAL_RATIO))
eval_ds = base.select(range(n_eval))
train_ds = base.select(range(n_eval, n))

def keep_example(ex: Dict[str, Any]) -> bool:
    r = ex.get(RESPONSE_FIELD)
    i = ex.get(INSTRUCTION_FIELD)
    return (r is not None and str(r).strip() != "") and (i is not None and str(i).strip() != "")

train_ds = train_ds.filter(keep_example)
eval_ds = eval_ds.filter(keep_example)

print("Rows total:", n)
print("Train:", len(train_ds), "Eval:", len(eval_ds))
print("Example raw_idx:", train_ds[0]["raw_idx"])
print("Example instruction:", train_ds[0][INSTRUCTION_FIELD][:160])

In [ ]:
# ---------- Model + tokenizer ----------
if not os.path.exists(BASE_MODEL_PATH):
    raise FileNotFoundError(f"Base model path not found: {BASE_MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_PATH)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    inference_mode=False,
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

if torch.cuda.is_available():
    model = model.to("cuda")

In [ ]:
# ---------- Tokenization ----------
token_cache_dir = f"{DIRS['shared_datasets_processed']}/{DATASET_NAME}__{BASE_MODEL_DIRNAME}__{TOKEN_CACHE_VERSION}"
train_tok_path = f"{token_cache_dir}/train"
eval_tok_path = f"{token_cache_dir}/eval"

def build_prompt(example: Dict[str, Any]) -> str:
    instr = (example.get(INSTRUCTION_FIELD) or "").strip()
    ctx = (example.get(CONTEXT_FIELD) or "").strip()

    if ctx:
        return f"### Instruction:\n{instr}\n\n### Context:\n{ctx}\n\n### Response:\n"
    return f"### Instruction:\n{instr}\n\n### Response:\n"

def preprocess_batch(batch: Dict[str, List[Any]]) -> Dict[str, Any]:
    prompts = []
    targets = []

    ctxs = batch.get(CONTEXT_FIELD, [""] * len(batch[INSTRUCTION_FIELD]))

    for instr, ctx, resp in zip(
        batch.get(INSTRUCTION_FIELD, []),
        ctxs,
        batch.get(RESPONSE_FIELD, []),
    ):
        instr = (instr or "").strip()
        ctx = (ctx or "").strip()
        resp = (resp or "").strip()

        prompt = (
            f"### Instruction:\n{instr}\n\n### Context:\n{ctx}\n\n### Response:\n"
            if ctx
            else f"### Instruction:\n{instr}\n\n### Response:\n"
        )

        prompts.append(prompt)
        targets.append(resp)

    model_inputs = tokenizer(
        prompts,
        max_length=MAX_SOURCE_LEN,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def add_raw_idx_batch(batch: Dict[str, List[Any]]) -> Dict[str, List[int]]:
    return {"raw_idx": [int(x) for x in batch["raw_idx"]]}

if CACHE_TOKENIZED and os.path.exists(train_tok_path) and os.path.exists(eval_tok_path):
    train_tok = load_from_disk(train_tok_path)
    eval_tok = load_from_disk(eval_tok_path)
    print("Loaded cached tokenized datasets.")
else:
    train_tok = train_ds.map(
        preprocess_batch,
        batched=True,
        remove_columns=train_ds.column_names,
        desc="Tokenizing train",
    )
    eval_tok = eval_ds.map(
        preprocess_batch,
        batched=True,
        remove_columns=eval_ds.column_names,
        desc="Tokenizing eval",
    )

    train_raw_idx = train_ds.map(
        add_raw_idx_batch,
        batched=True,
        remove_columns=[c for c in train_ds.column_names if c != "raw_idx"],
        desc="Extracting train raw_idx",
    )
    eval_raw_idx = eval_ds.map(
        add_raw_idx_batch,
        batched=True,
        remove_columns=[c for c in eval_ds.column_names if c != "raw_idx"],
        desc="Extracting eval raw_idx",
    )

    train_tok = train_tok.add_column("raw_idx", train_raw_idx["raw_idx"])
    eval_tok = eval_tok.add_column("raw_idx", eval_raw_idx["raw_idx"])

    if CACHE_TOKENIZED:
        _safe_makedirs(token_cache_dir)
        train_tok.save_to_disk(train_tok_path)
        eval_tok.save_to_disk(eval_tok_path)
        print("Saved tokenized datasets to cache.")

print(train_tok)
print(eval_tok)

In [ ]:
# ---------- Retrieval assets + sampler ----------
emb_path = f"{BUNDLE_DIR}/embeddings.npy"
neighbors_idx_path = f"{BUNDLE_DIR}/neighbors_topk_idx.npy"
neighbors_score_path = f"{BUNDLE_DIR}/neighbors_topk_scores.npy"

if SAMPLER_MODE in {"grouped", "mixed"}:
    for p in [emb_path, neighbors_idx_path, neighbors_score_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing retrieval asset: {p}")

    X = np.load(emb_path).astype("float32")
    X = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)
    neighbors_idx = np.load(neighbors_idx_path)
    neighbors_score = np.load(neighbors_score_path)

    trainpos_to_raw = [int(x) for x in train_tok["raw_idx"]]
    raw_to_trainpos = {r: i for i, r in enumerate(trainpos_to_raw)}

    print("Embeddings shape:", X.shape)
    print("Neighbors shape:", neighbors_idx.shape)
else:
    X = None
    neighbors_idx = None
    neighbors_score = None
    trainpos_to_raw = None
    raw_to_trainpos = None


@dataclass
class RetrievalBatchSamplerConfig:
    batch_size: int
    seed: int
    mode: str
    grouped_fraction: float = 0.80
    drop_last: bool = True

    def __post_init__(self):
        if self.mode not in {"grouped", "mixed"}:
            raise ValueError(f"Unsupported sampler mode: {self.mode}")
        if self.mode == "mixed" and not (0.0 < self.grouped_fraction < 1.0):
            raise ValueError("grouped_fraction must be in (0, 1) for mixed mode.")

    @property
    def grouped_target(self) -> int:
        if self.mode == "grouped":
            return self.batch_size
        return max(1, min(self.batch_size, int(round(self.batch_size * self.grouped_fraction))))

    @property
    def random_target(self) -> int:
        if self.mode == "grouped":
            return 0
        return max(0, self.batch_size - self.grouped_target)


class RetrievalBatchSampler:
    def __init__(
        self,
        trainpos_to_raw: List[int],
        raw_to_trainpos: Dict[int, int],
        neighbors_idx: np.ndarray,
        cfg: RetrievalBatchSamplerConfig,
    ):
        self.trainpos_to_raw = trainpos_to_raw
        self.raw_to_trainpos = raw_to_trainpos
        self.neighbors_idx = neighbors_idx
        self.cfg = cfg
        self.n = len(trainpos_to_raw)

    def __iter__(self):
        rng = random.Random(self.cfg.seed)
        anchors = list(range(self.n))
        rng.shuffle(anchors)

        all_positions = list(range(self.n))
        used_global = set()

        grouped_target = self.cfg.grouped_target
        batch_size = self.cfg.batch_size

        for a_pos in anchors:
            if a_pos in used_global:
                continue

            a_raw = self.trainpos_to_raw[a_pos]
            batch = [a_pos]
            used_in_batch = {a_pos}

            for nb_raw in self.neighbors_idx[a_raw]:
                if len(batch) >= grouped_target:
                    break

                nb_raw = int(nb_raw)
                nb_pos = self.raw_to_trainpos.get(nb_raw)

                if nb_pos is None or nb_pos in used_in_batch or nb_pos in used_global:
                    continue

                batch.append(nb_pos)
                used_in_batch.add(nb_pos)

            if self.cfg.mode == "mixed" and len(batch) < batch_size:
                candidates = [
                    p for p in all_positions
                    if p not in used_in_batch and p not in used_global
                ]
                rng.shuffle(candidates)

                needed = batch_size - len(batch)
                for p in candidates[:needed]:
                    batch.append(p)
                    used_in_batch.add(p)

            if len(batch) == batch_size:
                for p in batch:
                    used_global.add(p)
                yield batch
            elif not self.cfg.drop_last and len(batch) > 0:
                for p in batch:
                    used_global.add(p)
                yield batch

    def __len__(self):
        return self.n // self.cfg.batch_size


def build_train_batch_sampler(
    sampler_mode: str,
    batch_size: int,
    seed: int,
    trainpos_to_raw: Optional[List[int]],
    raw_to_trainpos: Optional[Dict[int, int]],
    neighbors_idx: Optional[np.ndarray],
    grouped_fraction: float = 0.80,
):
    if sampler_mode == "random":
        return None

    cfg = RetrievalBatchSamplerConfig(
        batch_size=batch_size,
        seed=seed,
        mode=sampler_mode,
        grouped_fraction=grouped_fraction,
        drop_last=True,
    )

    return RetrievalBatchSampler(
        trainpos_to_raw=trainpos_to_raw,
        raw_to_trainpos=raw_to_trainpos,
        neighbors_idx=neighbors_idx,
        cfg=cfg,
    )


def compute_coherence_for_batch(batch_trainpos: List[int]) -> float:
    a_pos = batch_trainpos[0]
    a_raw = trainpos_to_raw[a_pos]
    sims = [float(np.dot(X[a_raw], X[trainpos_to_raw[p]])) for p in batch_trainpos]
    return float(sum(sims) / len(sims))


batch_sampler = build_train_batch_sampler(
    sampler_mode=SAMPLER_MODE,
    batch_size=PER_DEVICE_BATCH,
    seed=SEED,
    trainpos_to_raw=trainpos_to_raw,
    raw_to_trainpos=raw_to_trainpos,
    neighbors_idx=neighbors_idx,
    grouped_fraction=GROUPED_FRACTION,
)

print("Custom batch sampler enabled:", batch_sampler is not None)

if batch_sampler is not None:
    first_batches = []
    for i, b in enumerate(batch_sampler):
        first_batches.append(b)
        if i >= 2:
            break

    print("Example sampled batches:", first_batches)
    print("Coherence:", [compute_coherence_for_batch(b) for b in first_batches])

    if SAMPLER_MODE == "mixed":
        print("Grouped target:", batch_sampler.cfg.grouped_target)
        print("Random target:", batch_sampler.cfg.random_target)

In [ ]:
# ---------- Trainer ----------
class JsonlLoggerCallback(TrainerCallback):
    def __init__(self, path: str):
        self.path = path
        _safe_makedirs(os.path.dirname(self.path))
        if os.path.exists(self.path):
            os.remove(self.path)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        row = {"step": int(state.global_step), "epoch": float(state.epoch or 0.0)}
        row.update({k: v for k, v in logs.items() if isinstance(v, (int, float))})
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row) + "\n")


class CustomBatchSeq2SeqTrainer(Seq2SeqTrainer):
    def __init__(self, *args, train_batch_sampler=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._train_batch_sampler = train_batch_sampler

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")

        if self._train_batch_sampler is None:
            return super().get_train_dataloader()

        return DataLoader(
            self.train_dataset,
            batch_sampler=self._train_batch_sampler,
            collate_fn=self.data_collator,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )


data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

steps_per_epoch = math.ceil(len(train_tok) / (PER_DEVICE_BATCH * GRAD_ACCUM))
max_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(max_steps * WARMUP_RATIO))

training_args = Seq2SeqTrainingArguments(
    output_dir=OUT_CHECKPOINTS,
    overwrite_output_dir=True,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    predict_with_generate=False,
    fp16=FP16,
    bf16=BF16,
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to=[],
    seed=SEED,
    data_seed=SEED,
    load_best_model_at_end=False,
)

trainer = CustomBatchSeq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
    callbacks=[JsonlLoggerCallback(METRICS_JSONL_PATH)],
    train_batch_sampler=batch_sampler,
)

In [ ]:
# ---------- Run manifest ----------
run_manifest = {
    "exp_id": EXP_ID,
    "method": METHOD,
    "created_at_utc": _now_utc(),
    "dataset_name": DATASET_NAME,
    "base_model": BASE_MODEL_DIRNAME,
    "seed": SEED,
    "train_size": len(train_tok),
    "eval_size": len(eval_tok),
    "bundle_name": BUNDLE_NAME,
    "hparams": {
        "epochs": EPOCHS,
        "per_device_batch": PER_DEVICE_BATCH,
        "grad_accum": GRAD_ACCUM,
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "warmup_steps": warmup_steps,
        "max_source_len": MAX_SOURCE_LEN,
        "max_target_len": MAX_TARGET_LEN,
    },
    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
    },
    "batching": {
        "sampler_mode": SAMPLER_MODE,
        "grouped_fraction": GROUPED_FRACTION if SAMPLER_MODE == "mixed" else None,
        "top_k": TOP_K if SAMPLER_MODE in {"grouped", "mixed"} else None,
        "batch_size": PER_DEVICE_BATCH,
        "sampler_seed": SEED,
        "bundle_dir": BUNDLE_DIR if SAMPLER_MODE in {"grouped", "mixed"} else None,
    },
    "paths": {
        "run_root": RUN_ROOT,
        "checkpoints": OUT_CHECKPOINTS,
        "logs": OUT_LOGS,
        "plots": OUT_PLOTS,
        "tables": OUT_TABLES,
        "adapter": OUT_ADAPTER,
        "metrics_jsonl": METRICS_JSONL_PATH,
        "loss_plot": LOSS_PLOT_PATH,
    },
}

with open(RUN_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, indent=2)

print("Wrote:", RUN_MANIFEST_PATH)

In [ ]:
# ---------- Train ----------
train_result = trainer.train()

eval_metrics = trainer.evaluate()
print("Train result:", train_result.metrics)
print("Eval metrics:", eval_metrics)

In [ ]:
# ---------- Save adapter + summarize ----------
trainer.model.save_pretrained(OUT_ADAPTER)
tokenizer.save_pretrained(OUT_ADAPTER)

metrics_rows = []
if os.path.exists(METRICS_JSONL_PATH):
    with open(METRICS_JSONL_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                metrics_rows.append(json.loads(line))

metrics_df = pd.DataFrame(metrics_rows)

loss_df = pd.DataFrame(columns=["step", "loss"])
if not metrics_df.empty and "loss" in metrics_df.columns:
    loss_df = metrics_df[metrics_df["loss"].notna()][["step", "loss"]].copy()
    loss_df = loss_df.sort_values("step").drop_duplicates(subset=["step"]).reset_index(drop=True)

coh_mean = None
coh_std = None
coh_n = 0
if batch_sampler is not None:
    cohs = []
    sampler_for_stats = build_train_batch_sampler(
        sampler_mode=SAMPLER_MODE,
        batch_size=PER_DEVICE_BATCH,
        seed=SEED,
        trainpos_to_raw=trainpos_to_raw,
        raw_to_trainpos=raw_to_trainpos,
        neighbors_idx=neighbors_idx,
        grouped_fraction=GROUPED_FRACTION,
    )
    for i, b in enumerate(sampler_for_stats):
        cohs.append(compute_coherence_for_batch(b))
        if i >= 500:
            break
    if cohs:
        coh_mean = float(np.mean(cohs))
        coh_std = float(np.std(cohs))
        coh_n = len(cohs)

if not loss_df.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(loss_df["step"], loss_df["loss"])
    plt.xlabel("Step")
    plt.ylabel("Train Loss")
    plt.title(f"Loss Curve - {EXP_ID}")
    plt.tight_layout()
    plt.savefig(LOSS_PLOT_PATH, dpi=150)
    plt.show()

train_loss_last = float(loss_df["loss"].iloc[-1]) if not loss_df.empty else None
eval_loss = float(eval_metrics["eval_loss"]) if "eval_loss" in eval_metrics else None

run_summary = {
    "exp_id": EXP_ID,
    "method": METHOD,
    "completed_at_utc": _now_utc(),
    "dataset_name": DATASET_NAME,
    "base_model": BASE_MODEL_DIRNAME,
    "sampler_mode": SAMPLER_MODE,
    "grouped_fraction": GROUPED_FRACTION if SAMPLER_MODE == "mixed" else None,
    "train_size": len(train_tok),
    "eval_size": len(eval_tok),
    "final_train_loss": train_loss_last,
    "final_eval_loss": eval_loss,
    "coherence_mean": coh_mean,
    "coherence_std": coh_std,
    "coherence_n_batches_sampled": coh_n,
    "paths": {
        "run_manifest": RUN_MANIFEST_PATH,
        "adapter": OUT_ADAPTER,
        "metrics_jsonl": METRICS_JSONL_PATH,
        "loss_plot": LOSS_PLOT_PATH,
    },
}

with open(RUN_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)

print("Wrote:", RUN_SUMMARY_PATH)
print(json.dumps(run_summary, indent=2))

In [ ]:
# ---------- Append lightweight experiment log ----------
row = {
    "created_at_utc": _now_utc(),
    "exp_id": EXP_ID,
    "method": METHOD,
    "dataset_name": DATASET_NAME,
    "base_model": BASE_MODEL_DIRNAME,
    "sampler_mode": SAMPLER_MODE,
    "grouped_fraction": GROUPED_FRACTION if SAMPLER_MODE == "mixed" else None,
    "final_train_loss": run_summary["final_train_loss"],
    "final_eval_loss": run_summary["final_eval_loss"],
    "coherence_mean": run_summary["coherence_mean"],
    "run_root": RUN_ROOT,
}

log_df = pd.DataFrame([row])
if os.path.exists(EXPERIMENT_CSV_PATH):
    existing = pd.read_csv(EXPERIMENT_CSV_PATH)
    combined = pd.concat([existing, log_df], ignore_index=True)
else:
    combined = log_df

combined.to_csv(EXPERIMENT_CSV_PATH, index=False)
print("Updated:", EXPERIMENT_CSV_PATH)
combined.tail()